# Pipeline ambx — Testes por Etapa

Este notebook executa e visualiza cada etapa da pipeline do framework **ambx** (*Ambient Access*), permitindo inspecionar os resultados intermediários antes de integrar tudo no script `pipeline.py`.

**Etapas:**
1. **1a — Malha Territorial**: geração da grade de análise (hexagonal/quadrada)
2. **1b — Pontos de Interesse**: coleta de POIs categorizados do OSM
3. **1c — Rede Viária**: construção do grafo de deslocamento
4. **1d — Vinculação Malha ↔ Rede**: snapping dos centróides aos nós do grafo
5. **2 — Roteamento A\*** : matriz origem-destino com caminhos mínimos (A* com heurística euclidiana)
6. **3 — Camadas Ambientais**: carregamento de camadas vetoriais (inundação, movimento de massa)
7. **4 — Penalidades** *(em implementação)*: aplicação de penalidades ambientais sobre as arestas

In [16]:
# --- Setup: path e imports ---
import sys
sys.path.insert(0, "../scripts")

import importlib

import ambx
from ambx.grid import generate_grid, GridFormat
from ambx.network import (
    add_travel_time,
    get_network,
    get_graph_edges,
    project_network,
    snap_grid_to_network,
)
from ambx.pois import get_pois
from ambx.routing import snap_pois_to_network, routing_matrix
from ambx.environment import build_environment, load_vector_from_gdf

import geopandas as gpd

print("✓ Todos os módulos importados com sucesso.")

✓ Todos os módulos importados com sucesso.


## Parâmetros

Defina aqui a localidade e os parâmetros da análise. Altere `LOCATION` para testar com outras cidades.

In [17]:
# --- Parâmetros da análise ---
LOCATION = "Porto Alegre, Rio Grande do Sul, Brazil"
GRID_FORMAT = GridFormat.HEXAGON
CELL_SIZE = 200        # metros (raio do hexágono ou lado do quadrado)
POI_BUFFER = 2000      # metros — captura serviços de cidades vizinhas
NETWORK_TYPE = "walk"  # walk, bike, drive
WALK_SPEED_KPH = 5.0   # km/h

print(f"Localidade : {LOCATION}")
print(f"Malha      : {GRID_FORMAT.value}, {CELL_SIZE}m")
print(f"Buffer POI : {POI_BUFFER}m")
print(f"Rede       : {NETWORK_TYPE} ({WALK_SPEED_KPH} km/h)")

Localidade : Porto Alegre, Rio Grande do Sul, Brazil
Malha      : hexagon, 200m
Buffer POI : 2000m
Rede       : walk (5.0 km/h)


## Etapa 1a — Malha Territorial

Geração da grade de células (hexágonos ou quadrados) que cobre o contorno administrativo da localidade.

In [18]:
# --- Gerar malha territorial ---
grid = generate_grid(LOCATION, grid_format=GRID_FORMAT, cell_size=CELL_SIZE)
print(f"Células geradas: {len(grid)}")
print(f"CRS: {grid.crs}")
print(f"Área total coberta: {grid.to_crs(grid.estimate_utm_crs()).area.sum() / 1e6:.2f} km²")

Células geradas: 5064
CRS: EPSG:4326
Área total coberta: 526.27 km²


## Etapa 1b — Pontos de Interesse

Coleta de destinos do OpenStreetMap categorizados em saúde, educação, transporte e alimentação.

In [19]:
# --- Coletar Pontos de Interesse ---
pois = get_pois(LOCATION, buffer=POI_BUFFER)
print(f"Total de POIs: {len(pois)}")
print(f"Categorias:\n{pois['category'].value_counts().to_string()}")

Total de POIs: 1764
Categorias:
category
education         713
food              502
health            477
transportation     72


## Etapa 1c — Rede Viária

Construção do grafo de deslocamento a partir do OpenStreetMap, projeção para CRS métrico e cálculo do tempo de percurso nas arestas.

In [20]:
# --- Construir grafo viário ---
print("Baixando grafo do OSM...")
graph = get_network(LOCATION, network_type=NETWORK_TYPE)
print(f"Nós (original): {graph.number_of_nodes()}")
print(f"Arestas (original): {graph.number_of_edges()}")

# Projetar para UTM
print("Projetando para UTM...")
graph = project_network(graph)
print(f"CRS do grafo projetado: {graph.graph.get('crs', 'N/A')}")

# Adicionar tempo de percurso
graph = add_travel_time(graph, speed_kph=WALK_SPEED_KPH)
print(f"Tempo de percurso adicionado (speed={WALK_SPEED_KPH} km/h)")

# Visualizar arestas
edges = get_graph_edges(graph)
print(f"Extensão total da rede: {edges.length.sum() / 1000:.1f} km")

Baixando grafo do OSM...
Nós (original): 41907
Arestas (original): 113542
Projetando para UTM...
CRS do grafo projetado: EPSG:32722
Tempo de percurso adicionado (speed=5.0 km/h)
Extensão total da rede: 8439.9 km


## Etapa 1d — Vinculação Malha ↔ Rede

Snapping dos centróides de cada célula da malha ao nó mais próximo do grafo viário. Células muito distantes da rede são descartadas.

In [21]:
# --- Vincular malha à rede ---
snapped = snap_grid_to_network(grid, graph, projected=False, max_distance=1000)

print(f"Células originais      : {len(grid)}")
print(f"Células vinculadas     : {len(snapped)}")
print(f"Taxa de vinculação     : {len(snapped) / len(grid) * 100:.1f}%")
print(f"Distância média snap   : {snapped['snap_distance'].mean():.1f} m")
print(f"Distância máxima snap  : {snapped['snap_distance'].max():.1f} m")

# Células não vinculadas (se houver limite de distância)
if len(snapped) < len(grid):
    lost = len(grid) - len(snapped)
    print(f"Células descartadas    : {lost} ({lost / len(grid) * 100:.1f}%)")

Células originais      : 5064
Células vinculadas     : 4357
Taxa de vinculação     : 86.0%
Distância média snap   : 172.8 m
Distância máxima snap  : 997.9 m
Células descartadas    : 707 (14.0%)


## Etapa 2 — Roteamento A*

Cálculo da matriz origem-destino: para cada célula da malha, encontra o caminho mínimo até cada POI usando o algoritmo **A\*** com heurística de distância euclidiana. O custo das arestas é o tempo de percurso (`travel_time`).

In [22]:
# --- Roteamento A*: K vizinhos mais próximos por categoria ---
import time

K_NEAREST = 3   # quantos POIs mais próximos por categoria considerar
N_JOBS = 38

print("Vinculando POIs à rede...")
pois_snapped = snap_pois_to_network(pois, graph)
print(f"POIs vinculados: {len(pois_snapped)} / {len(pois)}")
print(f"Distância média snap: {pois_snapped['snap_distance'].mean():.1f} m")

# Estatísticas de quantos pares serão calculados
cats = pois_snapped["category"].nunique()
est_pairs = len(snapped) * cats * K_NEAREST
print(f"\nEstimativa de pares A*: {len(snapped)} células × {cats} categorias × {K_NEAREST} = {est_pairs}")
print(f"(matriz completa seria: {len(snapped)} × {len(pois_snapped)} = {len(snapped) * len(pois_snapped)})")
print()

t0 = time.perf_counter()
# snapped_sample = snapped.sample(n=10, random_state=42)  # amostra para teste
matrix = routing_matrix(snapped, pois_snapped, graph, k_nearest=K_NEAREST, speed_kph=WALK_SPEED_KPH, n_jobs=N_JOBS)
elapsed = time.perf_counter() - t0

print(f"\n✓ Matriz concluída em {elapsed:.1f}s ({len(matrix) / elapsed:.0f} pares/s)")

# --- Resumo da matriz ---
reachable = matrix["travel_time"].notna().sum()
unreachable = matrix["travel_time"].isna().sum()
print(f"\nPares alcançáveis:   {reachable} ({reachable / len(matrix) * 100:.1f}%)")
print(f"Pares inalcançáveis: {unreachable} ({unreachable / len(matrix) * 100:.1f}%)")

print(f"\nTempo de viagem (alcançáveis):")
tt = matrix.loc[matrix["travel_time"].notna(), "travel_time"]
print(f"  Média:   {tt.mean() / 60:.1f} min")
print(f"  Mediana: {tt.median() / 60:.1f} min")
print(f"  Mínimo:  {tt.min() / 60:.1f} min")
print(f"  Máximo:  {tt.max() / 60:.1f} min")

print(f"\nDistância euclidiana vs tempo de viagem:")
print(f"  Correlação: {matrix['euclidean_dist'].corr(matrix['travel_time']):.3f}")

Vinculando POIs à rede...
POIs vinculados: 1764 / 1764
Distância média snap: 233.0 m

Estimativa de pares A*: 4357 células × 4 categorias × 3 = 52284
(matriz completa seria: 4357 × 1764 = 7685748)

  [seleção] célula 1/4357, 12 pares acumulados
  [seleção] célula 50/4357, 600 pares acumulados
  [seleção] célula 100/4357, 1200 pares acumulados
  [seleção] célula 150/4357, 1800 pares acumulados
  [seleção] célula 200/4357, 2400 pares acumulados
  [seleção] célula 250/4357, 3000 pares acumulados
  [seleção] célula 300/4357, 3600 pares acumulados
  [seleção] célula 350/4357, 4200 pares acumulados
  [seleção] célula 400/4357, 4800 pares acumulados
  [seleção] célula 450/4357, 5400 pares acumulados
  [seleção] célula 500/4357, 6000 pares acumulados
  [seleção] célula 550/4357, 6600 pares acumulados
  [seleção] célula 600/4357, 7200 pares acumulados
  [seleção] célula 650/4357, 7800 pares acumulados
  [seleção] célula 700/4357, 8400 pares acumulados
  [seleção] célula 750/4357, 9000 pares acu

## Mapa das Células Sample — Tempo Médio de Acesso

Visualização espacial das células amostradas, coloridas pelo tempo médio de acesso (média dos tempos de viagem A* para os POIs mais próximos de cada categoria).

In [23]:
# Calcular tempo médio de acesso por célula
avg_travel_time = (
    matrix.groupby("cell_idx")["travel_time"]
    .mean()
    .reset_index()
    .rename(columns={"travel_time": "avg_travel_time"})
)

# Pegar geometrias das células sample (do grid original)
# snapped_sample tem o mesmo index do grid original
cells = grid.loc[snapped.index].copy()
cells["cell_idx"] = cells.index

# Juntar com os tempos médios
cells_plot = cells.merge(avg_travel_time, on="cell_idx", how="left")

# cells_plot.explore(column="avg_travel_time", cmap="YlOrRd", legend=True)

## Etapa 3 — Camadas Ambientais

Carregamento de camadas vetoriais de risco (inundação, movimento de massa) a partir de arquivos GeoJSON. As camadas são recortadas automaticamente pela área de estudo e reprojetadas para o CRS UTM de referência.

In [24]:
# --- Carregar camadas ambientais vetoriais ---
importlib.reload(ambx.environment)

# Área de interesse: usamos o grid (GeoDataFrame) — o crs é extraído automaticamente
environment_layers = build_environment(
    area_of_interest=grid,
    vector_paths=[
        "../data/raw/porto_alegre/inundacao_cprm.geojson",
        "../data/raw/porto_alegre/movimento_massa_cprm.geojson"
    ],
)

# --- Diagnóstico das camadas carregadas ---
for layer in environment_layers.vectors:
    print(f"Camada vetorial: {layer.name}")
    print(f"  CRS: {layer.gdf.crs}")
    print(f"  Feições: {len(layer.gdf)}")
    print(f"  Colunas: {list(layer.gdf.columns)}")
    if 'classe' in layer.gdf.columns:
        print(f"  Valores únicos de classe: {layer.gdf['classe'].unique()}")
    print()

# --- Verificação de consistência de CRS ---
print("--- CRS de todos os elementos ---")
print(f"Grid (WGS84):          {grid.crs}")
print(f"POIs (WGS84):          {pois.crs}")
print(f"Graph (projetado):     {graph.graph.get('crs', 'N/A')}")
print(f"Env layers (UTM):      {environment_layers.vectors[0].gdf.crs}")
print(f"CRS UTM de referência: {environment_layers.crs_utm}")

# --- Visualização rápida das camadas sobre a malha ---
# Amostra algumas células para não sobrecarregar o mapa
grid_sample = grid.sample(n=min(500, len(grid)), random_state=42)

m = grid_sample.explore(
    style_kwds={"fill": None, "color": "gray", "weight": 0.5},
    name="Malha (amostra)",
)

for layer in environment_layers.vectors:
    m = layer.gdf.explore(
        m=m,
        color="red" if "inundacao" in layer.name else "orange",
        style_kwds={"fillOpacity": 0.3},
        name=layer.name,
    )

# m

Camada vetorial: inundacao_cprm
  CRS: EPSG:32722
  Feições: 2199
  Colunas: ['objectid', 'geometria', 'municipio', 'uf', 'processo', 'classe', 'obs', 'fonte', 'Shape__Area', 'Shape__Length', 'geometry']
  Valores únicos de classe: <ArrowStringArray>
['Médio', 'Alto', 'Baixa']
Length: 3, dtype: str

Camada vetorial: movimento_massa_cprm
  CRS: EPSG:32722
  Feições: 302
  Colunas: ['objectid', 'geomteria', 'municipio', 'uf', 'processo', 'classe', 'fonte', 'Shape__Area', 'Shape__Length', 'geometry']
  Valores únicos de classe: <ArrowStringArray>
['Média', 'Baixa', 'Alta']
Length: 3, dtype: str

--- CRS de todos os elementos ---
Grid (WGS84):          EPSG:4326
POIs (WGS84):          EPSG:4326
Graph (projetado):     EPSG:32722
Env layers (UTM):      EPSG:32722
CRS UTM de referência: EPSG:32722


## Etapa 4 — Penalidades (em implementação)

Aplicação das funções de penalização ambiental sobre os custos das arestas da rede. As regras de penalidade são definidas via `PenaltyRule` (fator multiplicador baseado no valor da camada) e aplicadas pelo módulo `penalties`.

### Testes:

1. **PenaltyRule** — validação do dataclass com diferentes tipos de função
2. **Risco sintético** — camada artificial com zonas de 3×3 km sobre a cidade, aplicada via `compose_penalties`

In [25]:
# --- Teste do PenaltyRule ---
importlib.reload(ambx)
importlib.reload(ambx.penalties)
from ambx.penalties import PenaltyRule

# 1. Teste com penalty_fn personalizada (faixas de inundação)
def flood_factor(depth: float) -> float:
    if depth > 50: return float("inf")
    if depth > 20: return 3.0
    if depth > 5:  return 1.5
    return 1.0

rule1 = PenaltyRule(
    layer_name="inundacao_cprm",
    layer_type="vector",
    penalty_fn=flood_factor,
)
print("Rule 1 (inundação):")
print(f"  flood_factor(10) = {rule1.penalty_fn(10)}")
print(f"  flood_factor(30) = {rule1.penalty_fn(30)}")
print(f"  flood_factor(60) = {rule1.penalty_fn(60)}")
print(f"  flood_factor(2)  = {rule1.penalty_fn(2)}")
print()

# 2. Teste com penalty_fn linear (temperatura)
rule2 = PenaltyRule(
    layer_name="temperatura_superficie",
    layer_type="raster",
    weight_field="travel_time",
    penalty_fn=lambda v: 1.0 + 0.1 * max(0, v - 30),
)
print("Rule 2 (temperatura — linear):")
print(f"  f(25°C) = {rule2.penalty_fn(25):.2f}")
print(f"  f(35°C) = {rule2.penalty_fn(35):.2f}")
print(f"  f(40°C) = {rule2.penalty_fn(40):.2f}")
print()

# 3. Teste com penalty_fn exponencial (declividade)
import math
rule3 = PenaltyRule(
    layer_name="declividade",
    layer_type="raster",
    weight_field="travel_time",
    penalty_fn=lambda v: math.exp(0.05 * v),
)
print("Rule 3 (declividade — exponencial):")
print(f"  f(0%)  = {rule3.penalty_fn(0):.2f}")
print(f"  f(10%) = {rule3.penalty_fn(10):.2f}")
print(f"  f(20%) = {rule3.penalty_fn(20):.2f}")
print()

# 4. Teste com valor default (sem penalidade)
rule4 = PenaltyRule(layer_name="risco_geologico", layer_type="vector")
print("Rule 4 (default — sem penalidade):")
print(f"  penalty_fn(999) = {rule4.penalty_fn(999)}")
print()

print("✓ Todos os testes de PenaltyRule passaram!")

Rule 1 (inundação):
  flood_factor(10) = 1.5
  flood_factor(30) = 3.0
  flood_factor(60) = inf
  flood_factor(2)  = 1.0

Rule 2 (temperatura — linear):
  f(25°C) = 1.00
  f(35°C) = 1.50
  f(40°C) = 2.00

Rule 3 (declividade — exponencial):
  f(0%)  = 1.00
  f(10%) = 1.65
  f(20%) = 2.72

Rule 4 (default — sem penalidade):
  penalty_fn(999) = 1.0

✓ Todos os testes de PenaltyRule passaram!


In [27]:
# --- Todas as camadas + compose_penalties ---
import numpy as np
from shapely.geometry import box

edges_gdf = edges

# CRS e bounds a partir das arestas já carregadas
edges_crs = edges_gdf.crs
bounds = grid.to_crs(edges_crs).total_bounds

# Malha de risco sintético de 3×3 km sobre a cidade
np.random.seed(42)
cell_size_m = 3000
x_start = np.floor(bounds[0] / cell_size_m) * cell_size_m
y_start = np.floor(bounds[1] / cell_size_m) * cell_size_m

poligonos, classes = [], []
x = x_start
while x < bounds[2]:
    y = y_start
    while y < bounds[3]:
        cell = box(x, y, x + cell_size_m, y + cell_size_m)
        if grid.to_crs(edges_crs).geometry.intersects(cell).any():
            poligonos.append(cell)
            r = np.random.random()
            if r < 0.05:      classes.append("Muito Alta")
            elif r < 0.15:    classes.append("Alta")
            elif r < 0.40:    classes.append("Média")
            else:             classes.append("Baixa")
        y += cell_size_m
    x += cell_size_m

gdf_risco = gpd.GeoDataFrame({"classe": classes, "geometry": poligonos}, crs=edges_crs)
risco_sintetico_layer = load_vector_from_gdf(
    gdf_risco, name="risco_sintetico", value_column="classe",
)
environment_layers.add_vector(risco_sintetico_layer)

print(f"Polígonos sintéticos: {len(gdf_risco)}")
print(f"Distribuição:\n{gdf_risco['classe'].value_counts()}")
print()

# --- Aplicar compose_penalties em todas as camadas ---
importlib.reload(ambx.penalties)
from ambx.penalties import compose_penalties, PenaltyRule

def risco_inundacao(v):
    return {"Alta": 5.0, "Média": 2.0, "Baixa": 1.2}.get(v, 1.0)

def risco_movimento(v):
    return {"Alta": 4.0, "Média": 1.8, "Baixa": 1.1}.get(v, 1.0)

def risco_sintetico(v):
    return {"Muito Alta": float("inf"), "Alta": 4.0, "Média": 1.8, "Baixa": 1.1}.get(v, 1.0)

# Prepara value_column de todas as camadas
for layer in environment_layers.vectors:
    layer.value_column = "classe"

# Cria uma regra para cada camada
rules = [
    PenaltyRule(layer.name, "vector", penalty_fn=fn)
    for layer, fn in zip(
        environment_layers.vectors,
        [risco_inundacao, risco_movimento, risco_sintetico],
    )
]

print("Regras:")
for r in rules:
    print(f"  • {r.layer_name}")

# Aplica compose
edges_final = compose_penalties(edges_gdf, environment_layers, rules)

# Resultados
tt_base = edges_gdf["travel_time"]
tt_final = edges_final["travel_time"]

modificadas = (tt_final > tt_base).sum()
interditadas = (tt_final == float("inf")).sum()

print(f"\n--- Resultado ---")
print(f"Arestas modificadas:  {modificadas} ({modificadas/len(edges_final)*100:.1f}%)")
print(f"Arestas interditadas: {interditadas}")

fatores = (tt_final[tt_final > tt_base] / tt_base[tt_final > tt_base].replace(0, np.nan))
fatores = fatores[~np.isinf(fatores)].dropna()
if len(fatores):
    print(f"Fator médio (não-inf): {fatores.mean():.2f}")
    print(f"Fatores únicos: {sorted(fatores.unique())}")

print("\n✓ compose_penalties aplicado com sucesso em todas as camadas!")

Polígonos sintéticos: 86
Distribuição:
classe
Baixa         43
Média         27
Alta          11
Muito Alta     5
Name: count, dtype: int64

Regras:
  • inundacao_cprm
  • movimento_massa_cprm
  • risco_sintetico

--- Resultado ---
Arestas modificadas:  113542 (100.0%)
Arestas interditadas: 402
Fator médio (não-inf): 1.97
Fatores únicos: [np.float64(1.0999999999999999), np.float64(1.1), np.float64(1.1000000000000003), np.float64(1.21), np.float64(1.2100000000000002), np.float64(1.2100000000000004), np.float64(1.4519999999999997), np.float64(1.452), np.float64(1.4520000000000002), np.float64(1.4520000000000004), np.float64(1.4520000000000006), np.float64(1.7999999999999998), np.float64(1.8), np.float64(1.8000000000000003), np.float64(1.9799999999999998), np.float64(1.98), np.float64(1.9800000000000002), np.float64(1.9800000000000004), np.float64(1.9800000000000006), np.float64(2.376), np.float64(2.3760000000000003), np.float64(2.376000000000001), np.float64(3.2399999999999998), np.float

In [29]:
# --- Roteamento: cenário típico vs condicionado ---
import time

# Já temos edges_gdf (arestas originais) e edges_final (arestas penalizadas)
print("Criando grafos para cada cenário...")

# Grafo base (já existe, com travel_time original)
graph_base = graph.copy()
print(f"Grafo base: {graph_base.number_of_edges()} arestas")

# Grafo condicionado: sobrescrever travel_time com os valores penalizados
graph_cond = graph.copy()

# Mapear (u, v, k) → travel_time penalizado
tt_penalized = edges_final["travel_time"]
for (u, v, k), tt in zip(edges_final.index, tt_penalized):
    if (u, v, k) in graph_cond.edges:
        graph_cond.edges[u, v, k]["travel_time"] = tt

print(f"Grafo condicionado: {graph_cond.number_of_edges()} arestas")
print()

# Verifica quantas arestas tiveram travel_time alterado
tt_orig = edges_gdf["travel_time"]
alteradas = (tt_penalized != tt_orig).sum()
print(f"Arestas com travel_time alterado: {alteradas} / {len(edges_gdf)}")
print()

# --- Roteamento no cenário condicionado ---
print("Roteando no cenário condicionado (A* com arestas penalizadas)...")
t0 = time.perf_counter()

matrix_cond = routing_matrix(
    snapped, pois_snapped, graph_cond,
    k_nearest=K_NEAREST, speed_kph=WALK_SPEED_KPH, n_jobs=N_JOBS,
)

elapsed = time.perf_counter() - t0
print(f"Matriz condicionada concluída em {elapsed:.1f}s")
print()

# --- Comparação dos dois cenários ---
print("--- Comparação: típico vs condicionado ---")

# A routing_matrix retorna poi_category (não category)
# Garantir que ambas têm o mesmo nome
col_category = "poi_category" if "poi_category" in matrix_cond.columns else "category"
if col_category not in matrix.columns:
    # matrix original pode ter sido renomeada — tenta o outro nome
    alt = "poi_category" if col_category == "category" else "category"
    if alt in matrix.columns:
        matrix = matrix.rename(columns={alt: col_category})

# Merge das duas matrizes
comp = matrix.merge(
    matrix_cond,
    on=["cell_idx", "poi_idx", col_category],
    suffixes=("_typ", "_cond"),
)

# Diferença de tempo
comp["delta_t"] = comp["travel_time_cond"] - comp["travel_time_typ"]
comp["delta_pct"] = (comp["delta_t"] / comp["travel_time_typ"].replace(0, float("nan"))) * 100

# Alcançabilidade
alv_typ = comp["travel_time_typ"].notna().sum()
alv_cond = comp["travel_time_cond"].notna().sum()
perdidos = (comp["travel_time_typ"].notna() & comp["travel_time_cond"].isna()).sum()

print(f"Pares alcançáveis (típico):       {alv_typ}")
print(f"Pares alcançáveis (condicionado): {alv_cond}")
print(f"Pares que se tornaram inalcançáveis: {perdidos}")
print()

# Estatísticas da diferença
delta = comp["delta_t"].dropna()
print(f"Aumento médio de tempo:   {delta.mean() / 60:.1f} min")
print(f"Aumento mediano de tempo: {delta.median() / 60:.1f} min")
print(f"Máximo aumento:           {delta.max() / 60:.1f} min")

if (delta > 0).sum() > 0:
    print(f"\nPares com aumento: {((delta > 0).sum())} ({(delta > 0).sum() / len(delta) * 100:.1f}%)")

print("\n✓ Comparação entre cenários concluída!")

Criando grafos para cada cenário...
Grafo base: 113542 arestas
Grafo condicionado: 113542 arestas

Arestas com travel_time alterado: 113542 / 113542

Roteando no cenário condicionado (A* com arestas penalizadas)...
  [seleção] célula 1/4357, 12 pares acumulados
  [seleção] célula 50/4357, 600 pares acumulados
  [seleção] célula 100/4357, 1200 pares acumulados
  [seleção] célula 150/4357, 1800 pares acumulados
  [seleção] célula 200/4357, 2400 pares acumulados
  [seleção] célula 250/4357, 3000 pares acumulados
  [seleção] célula 300/4357, 3600 pares acumulados
  [seleção] célula 350/4357, 4200 pares acumulados
  [seleção] célula 400/4357, 4800 pares acumulados
  [seleção] célula 450/4357, 5400 pares acumulados
  [seleção] célula 500/4357, 6000 pares acumulados
  [seleção] célula 550/4357, 6600 pares acumulados
  [seleção] célula 600/4357, 7200 pares acumulados
  [seleção] célula 650/4357, 7800 pares acumulados
  [seleção] célula 700/4357, 8400 pares acumulados
  [seleção] célula 750/435

## Resumo da Modelagem Territorial

Ao final destas etapas, temos todos os elementos necessários para a análise de acessibilidade:

| Elemento | Etapa | Descrição | Status |
|----------|:-----:|-----------|:------:|
| **Malha** | 1a | Grade de células de análise | ✅ |
| **POIs** | 1b | Destinos categorizados (saúde, educação, transporte, alimentação) | ✅ |
| **Rede** | 1c | Grafo viário com tempos de percurso | ✅ |
| **Snapping** | 1d | Mapeamento célula → nó da rede | ✅ |
| **Routing (A\*)** | 2 | Matriz origem-destino com caminhos mínimos | ✅ |
| **Camadas Ambientais** | 3 | Vetoriais (inundação, movimento de massa) carregadas e recortadas | ✅ |
| **Penalidades** | 4 | `PenaltyRule`, `apply_vector_penalty`, `compose_penalties` testados | ✅ |
| **Raster** | 4b | Pendente — `environment.raster` precisa ser validado primeiro | ❌ |
| **Indicadores** | — | Cálculo de PTh, Índice G e F15 | ❌ |
| **Comparação** | — | Análise comparativa entre cenários | ❌ |
| **Desigualdade** | — | Análise socioeconômica | ❌ |